# PointLLM Point Projector Fine-tuning

このノートブックでは、PointLLM-7B-v1.2のPoint Projector部分のみをファインチューニングします。

## アーキテクチャ概要

```
Point Cloud → [Point Encoder] → [Point Projector] → [LLM (LLaMA)]
   (N×6)        (凍結)            (学習対象)          (凍結)
              384次元出力        384→4096次元変換    4096次元入力
```

## 前提条件

- Google Colab A100 GPU環境
- Google Driveに学習データを配置済み
- PointLLMリポジトリをクローン済み

## 0. 環境構築（初回のみ）

以下のセルは環境構築用です。**初回実行時のみ**実行してください。
実行後、**ランタイムを再起動**する必要があります。

In [1]:
# 環境構築（初回のみ実行）
# 実行後はランタイムを再起動してください

SKIP_ENVIRONMENT_SETUP = True  # 環境構築済みの場合はTrue

if not SKIP_ENVIRONMENT_SETUP:
    import os
    import sys
    from google.colab import drive

    # 1. Driveマウント
    if not os.path.exists('/content/drive'):
        drive.mount('/content/drive')

    # 2. PyTorchを安定版 (2.4.1) に入れ替える
    print("=== PyTorch環境を 2.4.1 に統一します ===")
    !pip uninstall -y torch torchvision torchaudio
    !pip install torch==2.4.1 torchvision==0.19.1 torchaudio==2.4.1 --index-url https://download.pytorch.org/whl/cu121

    # 3. Flash Attention をインストール
    print("=== Flash Attention をインストールします ===")
    if os.path.exists("/content/drive/MyDrive/colab_wheels"):
        !pip install /content/drive/MyDrive/colab_wheels/flash_attn*.whl
    else:
        !pip install ninja packaging
        !pip install flash-attn==2.6.1 --no-build-isolation

    # 4. PointLLMのリポジトリ準備
    if not os.path.exists("/content/PointLLM"):
        !git clone https://github.com/InternRobotics/PointLLM
    %cd /content/PointLLM

    # 5. pyproject.toml を修正版に置き換え
    toml_content = '''[build-system]
requires = ["setuptools>=61.0"]
build-backend = "setuptools.build_meta"

[project]
name = "pointllm"
version = "0.1.2"
description = "Empower large language models to understand point clouds."
readme = "README.md"
requires-python = ">=3.8"
dependencies = [
    "accelerate",
    "einops",
    "fastapi",
    "gradio",
    "markdown2[all]",
    "numpy",
    "requests",
    "sentencepiece",
    "tokenizers==0.19.1",
    "uvicorn",
    "wandb",
    "shortuuid",
    "deepspeed",
    "peft==0.12.0",
    "transformers==4.44.2",
    "openai",
    "tqdm",
    "easydict",
    "timm==0.4.12",
    "ftfy==6.0.1",
    "regex",
    "open3d",
    "h5py",
    "termcolor",
    "plyfile",
    "nltk",
    "rouge",
    "scikit-learn",
    "py-rouge",
]
'''
    with open("pyproject.toml", "w") as f:
        f.write(toml_content)

    # 6. PointLLMのインストール
    !pip install ninja packaging
    !pip install -e .

    # 7. PointNet++ 演算モジュールのコンパイル
    if os.path.exists("pointnet2_ops_lib"):
        %cd pointnet2_ops_lib
        !pip install .
        %cd ..

    print("\n" + "="*50)
    print("✅ 環境構築が完了しました！")
    print("⚠️ 【重要】ランタイムを再起動してください")
    print("="*50)

---
## 1. セットアップ

ランタイム再起動後、以下のセルから実行を開始してください。

In [2]:
# パスとライブラリのセットアップ
import os
import sys
from pathlib import Path

# Google Driveをマウント
from google.colab import drive
if not os.path.exists('/content/drive'):
    drive.mount('/content/drive')

# パス設定
POINTLLM_PATH = "/content/drive/MyDrive/Colab Notebooks/研究/CustomPointLLM/PointLLM"
PROJECT_PATH = "/content/drive/MyDrive/Colab Notebooks/研究/CustomPointLLM/PointLLM/ProjectionFinetuning"

# パスを追加
for path in [POINTLLM_PATH, PROJECT_PATH]:
    if path not in sys.path:
        sys.path.insert(0, path)

# カレントディレクトリを変更
os.chdir(PROJECT_PATH)
print(f"Working directory: {os.getcwd()}")

# 基本ライブラリのインポート
import torch
import numpy as np
import json

# GPU確認
print(f"\nPyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"CUDA device: {torch.cuda.get_device_name(0)}")
    print(f"CUDA memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")

Mounted at /content/drive
Working directory: /content/drive/MyDrive/Colab Notebooks/研究/CustomPointLLM/PointLLM/ProjectionFinetuning

PyTorch version: 2.9.0+cu126
CUDA available: True
CUDA device: NVIDIA A100-SXM4-80GB
CUDA memory: 85.17 GB


## 2. 設定のカスタマイズ

学習パラメータとデータパスを設定します。

In [3]:
# 設定モジュールのインポート
from config import (
    FullConfig,
    create_default_config,
    create_memory_efficient_config,
    create_quick_test_config
)

# 設定の選択
# - create_default_config(): 標準設定（推奨）
# - create_memory_efficient_config(): メモリ節約設定
# - create_quick_test_config(): 動作確認用

config = create_default_config()
config.model.use_flash_attention = False
# ==================================================
# カスタム設定（必要に応じて変更）
# ==================================================

# データセットのパス
config.data.data_root = "/content/drive/MyDrive/Colab Notebooks/研究/CustomPointLLM/PointLLM/ProjectionFinetuning/llm_dataset"
config.data.train_annotation = "annotations/train.json"
config.data.val_annotation = "annotations/val.json"

# 学習パラメータ
config.training.num_epochs = 5          # エポック数
config.training.batch_size = 32          # バッチサイズ
config.training.gradient_accumulation_steps = 4  # 勾配累積
config.training.learning_rate = 5e-4    # 学習率

# 出力設定
config.output.experiment_name = "projection_finetune_v1"
config.output.output_dir = "/content/drive/MyDrive/Colab Notebooks/研究/CustomPointLLM/PointLLM/ProjectionFinetuning/outputs"
config.output.use_wandb = False  # WandBを使用する場合はTrue

# ==================================================

# 設定のサマリーを表示
config.print_summary()

Configuration Summary

[Model]
  Model: RunsenXu/PointLLM_7B_v1.2
  Dtype: bfloat16
  Freeze LLM: False
  Freeze Encoder: True

[Data]
  Root: /content/drive/MyDrive/Colab Notebooks/研究/CustomPointLLM/PointLLM/ProjectionFinetuning/llm_dataset
  Points: 8192
  Augmentation: True

[Training]
  Epochs: 5
  Batch size: 32
  Gradient accumulation: 4
  Effective batch size: 128
  Learning rate: 0.0005
  LR scheduler: cosine

[Output]
  Output dir: /content/drive/MyDrive/Colab Notebooks/研究/CustomPointLLM/PointLLM/ProjectionFinetuning/outputs/projection_finetune_v1
  WandB: False


## 3. データセットの準備

In [4]:
# データセットモジュールのインポート
from dataset import PointCloudProcessor, create_dataloaders
from model_utils import setup_pointllm_environment

# PointLLM環境のセットアップ
setup_pointllm_environment(POINTLLM_PATH)

# トークナイザーの読み込み
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained(
    config.model.model_name,
    use_fast=False,
    padding_side="right",
)

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
    tokenizer.pad_token_id = tokenizer.eos_token_id

print(f"Tokenizer loaded: {config.model.model_name}")
print(f"Vocab size: {len(tokenizer)}")

/usr/local/lib/python3.12/dist-packages/timm/models/layers/__init__.py:48: FutureWarning: Importing from timm.models.layers is deprecated, please import via timm.layers
  warnings.warn(f"Importing from {__name__} is deprecated, please import via timm.layers", FutureWarning)


✅ PointLLM environment setup complete


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/778 [00:00<?, ?B/s]

tokenizer.model:   0%|          | 0.00/500k [00:00<?, ?B/s]

added_tokens.json:   0%|          | 0.00/79.0 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/435 [00:00<?, ?B/s]

You are using the default legacy behaviour of the <class 'transformers.models.llama.tokenization_llama.LlamaTokenizer'>. This is expected, and simply means that the `legacy` (previous) behavior will be used so nothing changes for you. If you want to use the new behaviour, set `legacy=False`. This should only be set if you understand what it means, and thoroughly read the reason why this was added as explained in https://github.com/huggingface/transformers/pull/24565 - if you loaded a llama tokenizer from a GGUF file you can ignore this message


Tokenizer loaded: RunsenXu/PointLLM_7B_v1.2
Vocab size: 32003


In [5]:
# DataLoaderの作成
train_loader, val_loader = create_dataloaders(
    config=config.data,
    tokenizer=tokenizer,
    train_batch_size=config.training.batch_size,
    val_batch_size=config.training.batch_size * 2,
)

print(f"Train batches: {len(train_loader)}")
print(f"Val batches: {len(val_loader)}")

# サンプルバッチの確認
sample_batch = next(iter(train_loader))
print("\nSample batch shapes:")
for key, value in sample_batch.items():
    if isinstance(value, torch.Tensor):
        print(f"  {key}: {value.shape}")

Loaded 680 samples from /content/drive/MyDrive/Colab Notebooks/研究/CustomPointLLM/PointLLM/ProjectionFinetuning/llm_dataset/annotations/train.json
Loaded 84 samples from /content/drive/MyDrive/Colab Notebooks/研究/CustomPointLLM/PointLLM/ProjectionFinetuning/llm_dataset/annotations/val.json
Train loader: 21 batches
Val loader: 2 batches
Train batches: 21
Val batches: 2

Sample batch shapes:
  point_clouds: torch.Size([32, 8192, 6])
  input_ids: torch.Size([32, 2048])
  attention_mask: torch.Size([32, 2048])
  labels: torch.Size([32, 2048])


## 4. モデルの準備

In [6]:
# モデル読み込みモジュールのインポート
from model_utils import prepare_model_for_training, print_model_architecture
from peft import LoraConfig, get_peft_model, TaskType

# デバイス設定
device = "cuda" if torch.cuda.is_available() else "cpu"

# モデルの準備（読み込み + LLM/Encoder凍結）
model, tokenizer = prepare_model_for_training(
    config=config.model,
    pointllm_path=POINTLLM_PATH,
    device=device
)
# ✅ 追加: point_backbone_configの初期化
model.initialize_tokenizer_point_backbone_config_wo_embedding(tokenizer)

print("✅ Point backbone config initialized")
print(f"   point_patch_token: {model.get_model().point_backbone_config.get('point_patch_token')}")
print(f"   point_start_token: {model.get_model().point_backbone_config.get('point_start_token')}")
print(f"   point_end_token: {model.get_model().point_backbone_config.get('point_end_token')}")

# 1. LoRAの設定（LLM部分の学習設定）
lora_config = LoraConfig(
    r=8,                                # 低ランク行列のランク
    lora_alpha=32,                      # スケーリング係数
    target_modules=["q_proj", "v_proj"], # Vicuna(Llama)のSelf-Attention層を対象
    lora_dropout=0.05,
    bias="none",
    task_type=TaskType.CAUSAL_LM        # 因果言語モデル用
)

# 2. モデルにLoRAを適用
# この操作により、LLM本体の重みが固定され、LoRAアダプタが挿入されます。
# 同時に、既存のProjection層もデフォルトでフリーズされます。
model = get_peft_model(model, lora_config)

# 3. ✅ Projection層を明示的に学習対象（アンフリーズ）に設定
# PointLLMの内部構造におけるProjector（point_projector）を走査して有効化します。
for name, param in model.named_parameters():
    if "point_projector" in name:
        param.requires_grad = True

# 学習可能なパラメータ数を確認
model.print_trainable_parameters()
print("✅ LoRA adapters and Projection layer are now trainable.")

✅ PointLLM environment setup complete
Loading model: RunsenXu/PointLLM_7B_v1.2
✅ Special tokens already present in tokenizer
Token IDs:
  <point_patch>: 32000
  <point_start>: 32001
  <point_end>: 32002


config.json:   0%|          | 0.00/868 [00:00<?, ?B/s]

`torch_dtype` is deprecated! Use `dtype` instead!


pytorch_model.bin.index.json: 0.00B [00:00, ?B/s]

Fetching 3 files:   0%|          | 0/3 [00:00<?, ?it/s]

...

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Loading PointBERT config from /content/drive/MyDrive/Colab Notebooks/研究/CustomPointLLM/PointLLM/pointllm/model/pointbert/PointTransformer_8192point_2layer.yaml.


Loading checkpoint shards:   0%|          | 0/3 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/137 [00:00<?, ?B/s]

✅ Model loaded: PointLLMLlamaForCausalLM
   Dtype: torch.bfloat16
   Device: cuda
✅ Point backbone config initialized using built-in method

✅ Point Backbone Config (from inner model):
  point_token_len: 513
  mm_use_point_start_end: True
  point_patch_token ID: 32000
  point_start_token ID: 32001
  point_end_token ID: 32002

Parameter Freezing Summary
Total parameters: 6,771,185,536
Frozen parameters: 6,760,299,392 (99.84%)
Trainable parameters: 10,886,144 (0.16%)
------------------------------------------------------------
Trainable modules:
  model.point_proj.0.weight: 393,216 params
  model.point_proj.0.bias: 1,024 params
  model.point_proj.2.weight: 2,097,152 params
  model.point_proj.2.bias: 2,048 params
  model.point_proj.4.weight: 8,388,608 params
  model.point_proj.4.bias: 4,096 params

✅ Gradient checkpointing enabled for LLM backbone

Verifying Model Setup
✅ Point Projector found
✅ Point Projector trainable params: 10,886,144
✅ Point Encoder is frozen
✅ point_backbone_config

In [7]:
# GPUメモリ使用量の確認
if torch.cuda.is_available():
    print("GPU Memory Usage:")
    print(f"  Allocated: {torch.cuda.memory_allocated() / 1e9:.2f} GB")
    print(f"  Cached: {torch.cuda.memory_reserved() / 1e9:.2f} GB")

# モデルアーキテクチャの表示（オプション）
# print_model_architecture(model, max_depth=1)

GPU Memory Usage:
  Allocated: 13.56 GB
  Cached: 13.57 GB


## 5. 学習の実行

In [8]:
# WandBの設定（使用する場合）
if config.output.use_wandb:
    import wandb
    # APIキーの設定（必要に応じて）
    # wandb.login(key="YOUR_WANDB_API_KEY")

In [9]:
# 学習の実行
from trainer_LoRA import train_point_projector

history = train_point_projector(
    model=model,
    tokenizer=tokenizer,
    train_loader=train_loader,
    val_loader=val_loader,
    config=config,
    device=device
)

# 学習履歴を保存
history_path = Path(config.output.get_output_path()) / "training_history.json"
with open(history_path, 'w') as f:
    json.dump(history, f, indent=2)
print(f"Training history saved to: {history_path}")


Trainer Initialized
Total epochs: 5
Batch size: 32
Gradient accumulation: 4
Effective batch size: 128
Total steps: 25
Learning rate: 0.0005
Output directory: /content/drive/MyDrive/Colab Notebooks/研究/CustomPointLLM/PointLLM/ProjectionFinetuning/outputs/projection_finetune_v1


🚀 Starting Training



Epoch 1/5:   0%|          | 0/21 [00:00<?, ?it/s]/content/drive/MyDrive/Colab Notebooks/研究/CustomPointLLM/PointLLM/ProjectionFinetuning/trainer_LoRA.py:236: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast(enabled=use_amp, dtype=dtype):
Epoch 1/5: 100%|██████████| 21/21 [07:12<00:00, 20.58s/it, loss=1.9835, lr=4.79e-04]


Config saved to: /content/drive/MyDrive/Colab Notebooks/研究/CustomPointLLM/PointLLM/ProjectionFinetuning/outputs/projection_finetune_v1/checkpoints/checkpoint-epoch-1/config.json
💾 Checkpoint saved (including LoRA): /content/drive/MyDrive/Colab Notebooks/研究/CustomPointLLM/PointLLM/ProjectionFinetuning/outputs/projection_finetune_v1/checkpoints/checkpoint-epoch-1

📈 Epoch 1 completed: train_loss=2.6640



Epoch 2/5: 100%|██████████| 21/21 [06:58<00:00, 19.94s/it, loss=0.8416, lr=3.65e-04]


Config saved to: /content/drive/MyDrive/Colab Notebooks/研究/CustomPointLLM/PointLLM/ProjectionFinetuning/outputs/projection_finetune_v1/checkpoints/checkpoint-epoch-2/config.json
💾 Checkpoint saved (including LoRA): /content/drive/MyDrive/Colab Notebooks/研究/CustomPointLLM/PointLLM/ProjectionFinetuning/outputs/projection_finetune_v1/checkpoints/checkpoint-epoch-2

📈 Epoch 2 completed: train_loss=1.1227



...

Config saved to: /content/drive/MyDrive/Colab Notebooks/研究/CustomPointLLM/PointLLM/ProjectionFinetuning/outputs/projection_finetune_v1/checkpoints/checkpoint-epoch-3/config.json
💾 Checkpoint saved (including LoRA): /content/drive/MyDrive/Colab Notebooks/研究/CustomPointLLM/PointLLM/ProjectionFinetuning/outputs/projection_finetune_v1/checkpoints/checkpoint-epoch-3

📈 Epoch 3 completed: train_loss=0.5156



Epoch 4/5: 100%|██████████| 21/21 [06:58<00:00, 19.93s/it, loss=0.3396, lr=5.61e-05]


Config saved to: /content/drive/MyDrive/Colab Notebooks/研究/CustomPointLLM/PointLLM/ProjectionFinetuning/outputs/projection_finetune_v1/checkpoints/checkpoint-epoch-4/config.json
💾 Checkpoint saved (including LoRA): /content/drive/MyDrive/Colab Notebooks/研究/CustomPointLLM/PointLLM/ProjectionFinetuning/outputs/projection_finetune_v1/checkpoints/checkpoint-epoch-4
🗑️ Removed old checkpoint: /content/drive/MyDrive/Colab Notebooks/研究/CustomPointLLM/PointLLM/ProjectionFinetuning/outputs/projection_finetune_v1/checkpoints/checkpoint-epoch-1

📈 Epoch 4 completed: train_loss=0.3503



Epoch 5/5: 100%|██████████| 21/21 [06:58<00:00, 19.94s/it, loss=0.2430, lr=0.00e+00]


Config saved to: /content/drive/MyDrive/Colab Notebooks/研究/CustomPointLLM/PointLLM/ProjectionFinetuning/outputs/projection_finetune_v1/checkpoints/checkpoint-epoch-5/config.json
💾 Checkpoint saved (including LoRA): /content/drive/MyDrive/Colab Notebooks/研究/CustomPointLLM/PointLLM/ProjectionFinetuning/outputs/projection_finetune_v1/checkpoints/checkpoint-epoch-5
🗑️ Removed old checkpoint: /content/drive/MyDrive/Colab Notebooks/研究/CustomPointLLM/PointLLM/ProjectionFinetuning/outputs/projection_finetune_v1/checkpoints/checkpoint-epoch-2

📈 Epoch 5 completed: train_loss=0.2954


✅ Training Completed!
   Total time: 35.19 minutes
   Final step: 25
   Best validation loss: inf

Training history saved to: /content/drive/MyDrive/Colab Notebooks/研究/CustomPointLLM/PointLLM/ProjectionFinetuning/outputs/projection_finetune_v1/training_history.json


## 6. 評価と推論

In [10]:
# 評価モジュールのインポート
from evaluation import Evaluator, quick_inference
from model_utils import load_point_projector_weights

# ベストモデルの読み込み
best_checkpoint_path = Path(config.output.get_checkpoint_dir()) / "best_model"
if best_checkpoint_path.exists():
    model = load_point_projector_weights(
        model,
        str(best_checkpoint_path / "point_proj.pt"),
        strict=False
    )
    print(f"✅ Loaded best model from: {best_checkpoint_path}")

# ✅ モデル全体をbfloat16に変換
model = model.to(torch.bfloat16)
print("✅ Model converted to bfloat16")

# 評価クラスの初期化
evaluator = Evaluator(
    model=model,
    tokenizer=tokenizer,
    config=config,
    device=device
)

# パープレキシティの計算
perplexity = evaluator.compute_perplexity(val_loader)
print(f"\n📊 Final Perplexity: {perplexity:.4f}")

IndentationError: expected an indented block after 'for' statement on line 278 (evaluation.py, line 280)

In [ ]:
# 推論テスト

# テスト用の点群パスを設定（実際のパスに変更してください）
test_point_cloud = "/content/drive/MyDrive/Colab Notebooks/研究/CustomPointLLM/PointLLM/pointllm/data/objaverse_data/3c882707d08c4932b952eb4076d7a1b1_8192.npy"
test_question = "Descrive this point cloud 3D object."

if os.path.exists(test_point_cloud):
    response = quick_inference(
        model=model,
        tokenizer=tokenizer,
        point_cloud_path=test_point_cloud,
        question=test_question,
        config=config,
        device=device
    )

    print(f"Question: {test_question}")
    print(f"\nGenerated Response:")
    print("-" * 40)
    print(response)
    print("-" * 40)
else:
    print(f"⚠️ Test file not found: {test_point_cloud}")
    print("Please update the path to an existing point cloud file.")

## 7. カスタム推論

独自の点群ファイルと質問で推論を行います。

In [ ]:
# カスタム推論（パスと質問を変更して使用）

custom_point_cloud = "/path/to/your/point_cloud.npy"  # ← 変更してください
custom_question = "この3Dオブジェクトについて説明してください。"  # ← 変更してください

if os.path.exists(custom_point_cloud):
    response = quick_inference(
        model=model,
        tokenizer=tokenizer,
        point_cloud_path=custom_point_cloud,
        question=custom_question,
        config=config,
        device=device
    )
    print(f"Response: {response}")
else:
    print("Please set a valid point cloud path.")

## 8. クリーンアップ（オプション）

In [ ]:
# GPUメモリを解放
import gc

del model
del tokenizer
del train_loader
del val_loader

gc.collect()
torch.cuda.empty_cache()

print("✅ Cleanup complete")